# 05 — Final Evaluation, Operating Policy & SAS Handoff

> **Mục tiêu:** nghiệm thu model `logistic_core`, hiệu chỉnh xác suất, khóa threshold/decision policy trên validation, mở test đúng một lần về mặt quy trình và xuất scoring contract cho SAS.
>
> **Input:** artifact Notebook 04, `validation.parquet`, `test.parquet`, `feature_registry.csv`.
>
> **Output:** gói bất biến trong `models/transaction_fraud/final/` gồm pipeline, calibration, policy, test metrics, error analysis và SAS mapping.

Notebook này **không tạo feature mới, không chọn lại model, không tune bằng test**. Mọi quyết định vận hành được đóng băng trước khi đọc dữ liệu test.

## Table of Contents

1. [Contract và môi trường](#1)
2. [Nạp và khóa model đã chọn](#2)
3. [Kiểm tra và chọn calibration](#3)
4. [Risk score và threshold vận hành](#4)
5. [Decision bands](#5)
6. [Rule-only, Model-only và Hybrid](#6)
7. [Đóng băng policy trước test](#7)
8. [Mở holdout test đúng quy trình](#8)
9. [Metric cuối trên test](#9)
10. [Error analysis](#10)
11. [Ổn định theo nhóm](#11)
12. [Đóng gói bàn giao SAS](#12)

<a id="1"></a>
## 1. Contract và môi trường

Thiết lập đường dẫn, seed, giả định chi phí và design system chung. Chi phí dưới đây là **giả định mô phỏng**, không phải số liệu tài chính đã được phê duyệt.

In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import hashlib
import json
import shutil
import sys

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.calibration import calibration_curve
from sklearn.isotonic import IsotonicRegression
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    average_precision_score, brier_score_loss, confusion_matrix,
    f1_score, log_loss, precision_score, recall_score, roc_auc_score,
)
from sklearn.model_selection import train_test_split

REPO_ROOT = Path.cwd()
if not (REPO_ROOT / "notebooks").exists():
    REPO_ROOT = REPO_ROOT.parent
SRC_DIR = REPO_ROOT / "notebooks" / "src"
sys.path.insert(0, str(SRC_DIR.resolve()))

from scoring_utils import CalibratedFraudPipeline, probability_to_logit
from viz_utils import PALETTE, clean_ax, kpi_cards, setup

setup()
RANDOM_STATE = 42
MODEL_VERSION = "LOGISTIC_CORE_V1"
MODEL_DIR = REPO_ROOT / "models" / "transaction_fraud"
FINAL_DIR = MODEL_DIR / "final"
DATA_DIR = REPO_ROOT / "data" / "processed"

EXPECTED_MODEL = "logistic_core"
EXPECTED_TIER = "CORE"
EXPECTED_RAW_FEATURES = 34
CALIBRATION_FRACTION = 0.50
CAPACITIES_PER_1000 = [10, 20, 30, 50]
FIXED_RISK_SCORES = [50, 60, 70, 80, 90]
ANALYST_CAPACITY_PER_1000 = 30
MIN_WORST_SCENARIO_RECALL = 0.80
MAX_HARD_NEGATIVE_ALERT_RATE = 0.10
COST_PER_ALERT_VND = 50_000
MISSED_FRAUD_LOSS_MULTIPLIER = 1.0

print(f"Repository : {REPO_ROOT}")
print(f"Final dir  : {FINAL_DIR}")
print("Test policy: chưa đọc test.parquet trong các section 1–7")

### Helper đánh giá và kiểm soát policy

Các hàm dưới đây chuẩn hóa weighted metric, ECE, recall theo scenario, ngưỡng theo capacity và checksum. Chúng chỉ đo lường; không fit hoặc thay đổi model gốc.

In [ ]:
def sha256_file(path):
    digest = hashlib.sha256()
    with open(path, "rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


def weighted_ece(y_true, score, sample_weight=None, n_bins=10):
    y_true = np.asarray(y_true, dtype=float)
    score = np.asarray(score, dtype=float)
    weight = np.ones(len(y_true)) if sample_weight is None else np.asarray(sample_weight, dtype=float)
    edges = np.linspace(0.0, 1.0, n_bins + 1)
    bucket = np.clip(np.digitize(score, edges[1:-1], right=True), 0, n_bins - 1)
    total = weight.sum()
    value = 0.0
    for bin_id in range(n_bins):
        mask = bucket == bin_id
        if mask.any():
            w = weight[mask]
            value += (w.sum() / total) * abs(np.average(y_true[mask], weights=w) - np.average(score[mask], weights=w))
    return float(value)


def safe_auc(metric, y_true, score, sample_weight=None):
    return float(metric(y_true, score, sample_weight=sample_weight)) if pd.Series(y_true).nunique() == 2 else np.nan


def scenario_recall_table(frame, prediction_col):
    fraud = frame.loc[frame["target_fraud"].eq(1) & frame["scenario_code"].ne("BACKGROUND")].copy()
    rows = []
    for scenario, group in fraud.groupby("scenario_code", dropna=False):
        rows.append({
            "scenario_code": scenario,
            "fraud_transactions": len(group),
            "event_weight": group["sample_weight"].sum(),
            "recall": np.average(group[prediction_col], weights=group["sample_weight"]),
        })
    return pd.DataFrame(rows).sort_values("scenario_code").reset_index(drop=True)


def policy_metrics(frame, score_col, prediction, policy_name, threshold=np.nan):
    y = frame["target_fraud"].astype(int).to_numpy()
    score = frame[score_col].to_numpy(dtype=float)
    pred = np.asarray(prediction, dtype=int)
    weight = frame["sample_weight"].to_numpy(dtype=float)
    scenario = frame.assign(_prediction=pred)
    scenario_table = scenario_recall_table(scenario, "_prediction")
    hard_negative = frame["hard_negative"].astype(int).eq(1).to_numpy()
    fraud_amount = frame.loc[frame["target_fraud"].eq(1), "amount_num"].sum()
    captured_amount = frame.loc[(frame["target_fraud"].eq(1)) & (pred == 1), "amount_num"].sum()
    false_negative_amount = max(fraud_amount - captured_amount, 0.0)
    expected_cost = pred.sum() * COST_PER_ALERT_VND + false_negative_amount * MISSED_FRAUD_LOSS_MULTIPLIER
    return {
        "policy": policy_name,
        "threshold": float(threshold) if pd.notna(threshold) else np.nan,
        "alerts": int(pred.sum()),
        "alerts_per_1000": float(1000 * pred.mean()),
        "precision_event_weighted": precision_score(y, pred, sample_weight=weight, zero_division=0),
        "recall_event_weighted": recall_score(y, pred, sample_weight=weight, zero_division=0),
        "f1_event_weighted": f1_score(y, pred, sample_weight=weight, zero_division=0),
        "hard_negative_alert_rate": float(pred[hard_negative].mean()) if hard_negative.any() else np.nan,
        "worst_scenario_recall": float(scenario_table["recall"].min()) if len(scenario_table) else np.nan,
        "fraud_amount_captured": float(captured_amount),
        "fraud_amount_capture_rate": float(captured_amount / fraud_amount) if fraud_amount else np.nan,
        "expected_cost_vnd": float(expected_cost),
    }


def threshold_for_capacity(scores, alerts_per_1000):
    scores = np.asarray(scores, dtype=float)
    k = max(1, int(np.ceil(len(scores) * alerts_per_1000 / 1000)))
    return float(np.partition(scores, len(scores) - k)[len(scores) - k])


def exact_capacity_prediction(frame, score_col, alerts_per_1000):
    # Exact top-k mask with a stable transaction-id tie break.
    k = max(1, int(np.ceil(len(frame) * alerts_per_1000 / 1000)))
    ranked = frame[[score_col, "transaction_id"]].sort_values(
        [score_col, "transaction_id"], ascending=[False, True], kind="mergesort"
    )
    selected_index = ranked.index[:k]
    prediction = pd.Series(0, index=frame.index, dtype=int)
    prediction.loc[selected_index] = 1
    return prediction


def calibration_metrics(name, y_true, score, sample_weight):
    return {
        "method": name,
        "brier_score": brier_score_loss(y_true, score, sample_weight=sample_weight),
        "log_loss": log_loss(y_true, np.column_stack([1-score, score]), sample_weight=sample_weight, labels=[0, 1]),
        "ece_10_bins": weighted_ece(y_true, score, sample_weight, n_bins=10),
        "pr_auc_event_weighted": average_precision_score(y_true, score, sample_weight=sample_weight),
    }

<a id="2"></a>
## 2. Nạp và khóa model đã chọn

Chỉ nạp artifact phát triển từ Notebook 04 và validation. Cell này xác nhận đúng `logistic_core`, tier CORE, 34 raw feature, pipeline sklearn và trạng thái `test_opened=false`; `test.parquet` mới chỉ được kiểm tra tồn tại bằng metadata filesystem.

In [ ]:
selected_path = MODEL_DIR / "selected_pipeline.joblib"
manifest_path = MODEL_DIR / "training_manifest.json"
validation_prediction_path = MODEL_DIR / "validation_predictions.parquet"
validation_capacity_path = MODEL_DIR / "validation_alert_capacity.csv"
registry_path = DATA_DIR / "feature_registry.csv"
split_manifest_path = DATA_DIR / "split_manifest.json"
validation_path = DATA_DIR / "validation.parquet"
test_path = DATA_DIR / "test.parquet"

required_pretest = [selected_path, manifest_path, validation_prediction_path, validation_capacity_path, registry_path, split_manifest_path, validation_path]
missing = [str(path) for path in required_pretest if not path.exists()]
assert not missing, f"Thiếu artifact upstream: {missing}"
assert test_path.exists(), "Thiếu test.parquet; chỉ kiểm tra path, chưa đọc dữ liệu."

training_manifest = json.loads(manifest_path.read_text(encoding="utf-8"))
split_manifest = json.loads(split_manifest_path.read_text(encoding="utf-8"))
selected_pipeline = joblib.load(selected_path)
feature_registry = pd.read_csv(registry_path)
df_validation = pd.read_parquet(validation_path)
validation_predictions_nb04 = pd.read_parquet(validation_prediction_path)
validation_alert_capacity_nb04 = pd.read_csv(validation_capacity_path)

core_features = split_manifest["feature_subsets"]["CORE_FEATURES"]["columns"]
assert training_manifest["selection_status"] == "CHAMPION", "Notebook 04 chưa tạo champion chính thức."
assert training_manifest["selected_experiment"] == EXPECTED_MODEL
assert training_manifest["selected_tier"] == EXPECTED_TIER
assert training_manifest["test_opened"] is False
assert len(core_features) == EXPECTED_RAW_FEATURES == training_manifest["selected_raw_feature_count"]
assert list(selected_pipeline.named_steps) == ["preprocessor", "model"]
assert isinstance(selected_pipeline.named_steps["model"], LogisticRegression)
assert not (set(core_features) - set(feature_registry["feature_name"]))
assert not (set(core_features) - set(df_validation.columns))

required_audit = {"transaction_id", "account_id", "customer_id", "simulation_run_id", "scenario_code", "sample_weight", "hard_negative", "amount_num", "beneficiary_id", "txn_time_utc"}
assert not (required_audit - set(df_validation.columns)), f"Validation thiếu audit-only fields: {required_audit - set(df_validation.columns)}"

raw_validation_score = selected_pipeline.predict_proba(df_validation[core_features])[:, 1]
exported = validation_predictions_nb04.set_index("transaction_id")["fraud_score_uncalibrated"]
aligned_exported = df_validation["transaction_id"].map(exported).to_numpy()
assert np.allclose(raw_validation_score, aligned_exported, atol=1e-12), "Score validation không khớp artifact Notebook 04."

model_lock = {
    "model": EXPECTED_MODEL,
    "tier": EXPECTED_TIER,
    "feature_count": len(core_features),
    "pipeline_sha256": sha256_file(selected_path),
    "feature_order_sha256": hashlib.sha256("\n".join(core_features).encode()).hexdigest(),
    "test_opened": False,
}
kpi_cards([
    {"label": "Selected model", "value": EXPECTED_MODEL, "color": PALETTE["primary"]},
    {"label": "Raw features", "value": str(len(core_features)), "color": PALETTE["blue"]},
    {"label": "Validation rows", "value": f"{len(df_validation):,}", "color": PALETTE["accent"]},
    {"label": "Test opened", "value": "NO", "color": PALETTE["stable"]},
])

<a id="3"></a>
## 3. Kiểm tra và chọn calibration

Validation được tách theo `customer_id`: một nửa fit calibrator, nửa còn lại chọn calibration và policy. Base pipeline không fit lại. Platt và isotonic được so với raw probability bằng Brier, log loss và ECE; PR-AUC chỉ xác nhận thứ hạng không bị phá vỡ.

In [ ]:
entity_label = df_validation.groupby("customer_id")["target_fraud"].max()
calibration_customers, policy_customers = train_test_split(
    entity_label.index,
    test_size=1.0 - CALIBRATION_FRACTION,
    random_state=RANDOM_STATE,
    stratify=entity_label.to_numpy(),
)
calibration_mask = df_validation["customer_id"].isin(calibration_customers)
policy_mask = df_validation["customer_id"].isin(policy_customers)
assert not set(calibration_customers).intersection(policy_customers)

df_calibration = df_validation.loc[calibration_mask].copy()
df_policy = df_validation.loc[policy_mask].copy()
raw_calibration = raw_validation_score[calibration_mask]
raw_policy = raw_validation_score[policy_mask]
y_calibration = df_calibration["target_fraud"].astype(int).to_numpy()
y_policy = df_policy["target_fraud"].astype(int).to_numpy()
w_calibration = df_calibration["sample_weight"].to_numpy(dtype=float)
w_policy = df_policy["sample_weight"].to_numpy(dtype=float)

platt = LogisticRegression(C=1.0, solver="lbfgs", random_state=RANDOM_STATE)
platt.fit(probability_to_logit(raw_calibration), y_calibration, sample_weight=w_calibration)

calibrators = {"raw": None, "platt": platt}
if len(df_calibration) >= 1_000 and y_calibration.sum() >= 50 and (len(y_calibration) - y_calibration.sum()) >= 50:
    isotonic = IsotonicRegression(out_of_bounds="clip")
    isotonic.fit(probability_to_logit(raw_calibration).ravel(), y_calibration, sample_weight=w_calibration)
    calibrators["isotonic"] = isotonic


def apply_calibrator(method, calibrator, raw_score):
    if method == "raw":
        return np.asarray(raw_score)
    logits = probability_to_logit(raw_score)
    if method == "platt":
        return calibrator.predict_proba(logits)[:, 1]
    return calibrator.predict(logits.ravel())


policy_scores = {name: apply_calibrator(name, cal, raw_policy) for name, cal in calibrators.items()}
calibration_comparison = pd.DataFrame([
    calibration_metrics(name, y_policy, score, w_policy)
    for name, score in policy_scores.items()
]).sort_values(["brier_score", "log_loss", "ece_10_bins"]).reset_index(drop=True)

selected_calibration_method = calibration_comparison.loc[0, "method"]
selected_calibrator = calibrators[selected_calibration_method]
df_policy["fraud_score_uncalibrated"] = raw_policy
df_policy["fraud_probability"] = policy_scores[selected_calibration_method]
display(calibration_comparison.style.format({
    "brier_score": "{:.6f}", "log_loss": "{:.6f}", "ece_10_bins": "{:.6f}", "pr_auc_event_weighted": "{:.6f}"
}))
print(f"Calibration được chọn trên policy-validation: {selected_calibration_method}")

### Calibration curve — score có thể được gọi là xác suất hay chưa?

Đường càng gần đường chéo thì xác suất dự báo càng sát tỷ lệ fraud quan sát. Nếu chỉ dùng score để xếp hạng, raw score vẫn hữu ích; khi xuất `fraud_probability`, calibration được chọn là contract chính thức.

In [ ]:
fig, ax = plt.subplots(figsize=(8.5, 5.2))
ax.plot([0, 1], [0, 1], linestyle="--", color=PALETTE["neutral"], label="Perfect calibration")
for idx, (name, score) in enumerate(policy_scores.items()):
    observed, predicted = calibration_curve(y_policy, score, n_bins=10, strategy="quantile")
    ax.plot(predicted, observed, marker="o", color=list(PALETTE.values())[idx], label=name)
clean_ax(ax, "Calibration trên policy-validation", "Mean predicted probability", "Observed fraud rate", "Chọn phương pháp bằng Brier/log loss/ECE trên phần validation không fit calibrator")
ax.legend()
plt.show()

<a id="4"></a>
## 4. Risk score và threshold vận hành

Quy đổi `model_risk_score = 100 × fraud_probability`. Notebook đánh giá bốn mức capacity và năm ngưỡng score. Threshold được chọn trên policy-validation theo capacity analyst, recall scenario, hard-negative rate và expected cost giả định.

In [ ]:
df_policy["model_risk_score"] = 100.0 * df_policy["fraud_probability"]

capacity_rows = []
for capacity in CAPACITIES_PER_1000:
    threshold = threshold_for_capacity(df_policy["fraud_probability"], capacity)
    pred = df_policy["fraud_probability"].ge(threshold).astype(int)
    row = policy_metrics(df_policy, "fraud_probability", pred, f"capacity_{capacity}_per_1000", threshold)
    row["candidate_type"] = "capacity"
    row["requested_alerts_per_1000"] = capacity
    capacity_rows.append(row)

fixed_rows = []
for risk_score in FIXED_RISK_SCORES:
    threshold = risk_score / 100.0
    pred = df_policy["fraud_probability"].ge(threshold).astype(int)
    row = policy_metrics(df_policy, "fraud_probability", pred, f"risk_score_ge_{risk_score}", threshold)
    row["candidate_type"] = "fixed_score"
    row["requested_alerts_per_1000"] = np.nan
    fixed_rows.append(row)

threshold_candidates = pd.DataFrame(capacity_rows + fixed_rows)
eligible_policy = threshold_candidates.loc[
    threshold_candidates["candidate_type"].eq("capacity")
    & threshold_candidates["alerts_per_1000"].le(ANALYST_CAPACITY_PER_1000 + 0.5)
    & threshold_candidates["worst_scenario_recall"].ge(MIN_WORST_SCENARIO_RECALL)
    & threshold_candidates["hard_negative_alert_rate"].le(MAX_HARD_NEGATIVE_ALERT_RATE)
].copy()

policy_selected_under_relaxed_gate = eligible_policy.empty
if policy_selected_under_relaxed_gate:
    eligible_policy = threshold_candidates.loc[
        threshold_candidates["candidate_type"].eq("capacity")
        & threshold_candidates["alerts_per_1000"].le(ANALYST_CAPACITY_PER_1000 + 0.5)
    ].copy()
    assert not eligible_policy.empty, "Không có threshold nào nằm trong capacity analyst."

selected_policy_row = eligible_policy.sort_values(
    ["expected_cost_vnd", "worst_scenario_recall", "hard_negative_alert_rate"],
    ascending=[True, False, True],
).iloc[0]
selected_threshold = float(selected_policy_row["threshold"])
df_policy["model_alert"] = df_policy["fraud_probability"].ge(selected_threshold).astype(int)

display(threshold_candidates.style.format({
    "threshold": "{:.6f}", "alerts_per_1000": "{:.2f}",
    "precision_event_weighted": "{:.2%}", "recall_event_weighted": "{:.2%}",
    "hard_negative_alert_rate": "{:.2%}", "worst_scenario_recall": "{:.2%}",
    "fraud_amount_capture_rate": "{:.2%}", "expected_cost_vnd": "{:,.0f}",
}))
print(f"Threshold đã chọn: {selected_threshold:.6f} | relaxed gate: {policy_selected_under_relaxed_gate}")

### Trade-off capacity — thêm alert mua được bao nhiêu recall?

Biểu đồ đặt recall và hard-negative alert rate trên cùng trục capacity. Đây là bằng chứng để analyst capacity trở thành quyết định vận hành, không phải ngưỡng mặc định 3%.

In [ ]:
capacity_view = threshold_candidates.query("candidate_type == 'capacity'").sort_values("alerts_per_1000")
fig, ax = plt.subplots(figsize=(8.5, 5.2))
ax.plot(capacity_view["alerts_per_1000"], capacity_view["recall_event_weighted"], marker="o", color=PALETTE["danger"], label="Fraud recall")
ax.plot(capacity_view["alerts_per_1000"], capacity_view["hard_negative_alert_rate"], marker="o", color=PALETTE["accent"], label="Hard-negative alert rate")
ax.axvline(ANALYST_CAPACITY_PER_1000, linestyle="--", color=PALETTE["neutral"], label="Analyst capacity")
clean_ax(ax, "Capacity policy trên validation", "Alerts / 1.000 transactions", "Rate", "Threshold cuối được chọn trước khi test được mở")
ax.yaxis.set_major_formatter(lambda x, pos: f"{x:.0%}")
ax.legend()
plt.show()

<a id="5"></a>
## 5. Decision bands

Các band được suy ra từ quantile/capacity trên policy-validation: Medium tương ứng top 50/1.000, High là threshold vận hành, Critical là top 10/1.000. Không đặt ngưỡng tùy ý sau khi xem test.

In [ ]:
threshold_by_capacity = capacity_view.set_index("requested_alerts_per_1000")["threshold"].to_dict()
medium_threshold = float(threshold_by_capacity[50])
high_threshold = selected_threshold
critical_threshold = float(threshold_by_capacity[10])
assert 0 <= medium_threshold <= high_threshold <= critical_threshold <= 1

decision_bands = pd.DataFrame([
    {"band": "LOW", "min_probability": 0.0, "max_probability_exclusive": medium_threshold, "min_risk_score": 0.0, "action": "APPROVE"},
    {"band": "MEDIUM", "min_probability": medium_threshold, "max_probability_exclusive": high_threshold, "min_risk_score": 100*medium_threshold, "action": "REVIEW_CONTEXT"},
    {"band": "HIGH", "min_probability": high_threshold, "max_probability_exclusive": critical_threshold, "min_risk_score": 100*high_threshold, "action": "CHALLENGE_OR_ALERT"},
    {"band": "CRITICAL", "min_probability": critical_threshold, "max_probability_exclusive": 1.000001, "min_risk_score": 100*critical_threshold, "action": "HOLD_AND_ALERT"},
])


def assign_band(probability):
    return pd.cut(
        probability,
        bins=[-np.inf, medium_threshold, high_threshold, critical_threshold, np.inf],
        labels=["LOW", "MEDIUM", "HIGH", "CRITICAL"],
        right=False,
    ).astype(str)


df_policy["risk_band"] = assign_band(df_policy["fraud_probability"])
display(decision_bands.style.format({"min_probability": "{:.6f}", "max_probability_exclusive": "{:.6f}", "min_risk_score": "{:.2f}"}))

<a id="6"></a>
## 6. Rule-only, Model-only và Hybrid

Rule baseline và hybrid chỉ dùng tín hiệu CORE có tại thời điểm T. `hard_rule_alert` ở đây là policy decision, không phải feature mới. Ba chiến lược được so trên cùng policy-validation và cùng metric contract.

In [ ]:
def operational_rule_score(frame):
    components = [
        frame["is_new_device"].astype(int),
        frame["is_new_beneficiary"].astype(int),
        frame["is_high_balance_drain"].astype(int),
        frame["is_high_limit_usage"].astype(int),
        frame["is_external_transfer"].astype(int),
        frame["is_night"].astype(int),
    ]
    return sum(components) / len(components)


df_policy["business_rule_score"] = operational_rule_score(df_policy)
rule_capacity = float(selected_policy_row["alerts_per_1000"])
rule_threshold = threshold_for_capacity(df_policy["business_rule_score"], rule_capacity)
df_policy["rule_alert"] = exact_capacity_prediction(df_policy, "business_rule_score", rule_capacity)
df_policy["hard_rule_alert"] = (
    df_policy["is_new_device"].astype(bool)
    & df_policy["is_new_beneficiary"].astype(bool)
    & (df_policy["is_high_balance_drain"].astype(bool) | df_policy["is_high_limit_usage"].astype(bool))
).astype(int)
df_policy["hybrid_alert"] = (df_policy["model_alert"].eq(1) | df_policy["hard_rule_alert"].eq(1)).astype(int)

strategy_comparison = pd.DataFrame([
    policy_metrics(df_policy, "business_rule_score", df_policy["rule_alert"], "rule_only", rule_threshold),
    policy_metrics(df_policy, "fraud_probability", df_policy["model_alert"], "model_only", selected_threshold),
    policy_metrics(df_policy, "fraud_probability", df_policy["hybrid_alert"], "hybrid", selected_threshold),
])
display(strategy_comparison.style.format({
    "threshold": "{:.6f}", "alerts_per_1000": "{:.2f}",
    "precision_event_weighted": "{:.2%}", "recall_event_weighted": "{:.2%}",
    "hard_negative_alert_rate": "{:.2%}", "worst_scenario_recall": "{:.2%}",
    "fraud_amount_capture_rate": "{:.2%}", "expected_cost_vnd": "{:,.0f}",
}))

<a id="7"></a>
## 7. Đóng băng policy trước test

Cell này tạo scorer cuối, lưu calibrator, threshold, bands, feature contract và manifest khóa với `test_opened=false`. Sau cell này không còn thao tác fit, chọn feature, chọn model, chọn calibrator hoặc đổi threshold.

In [ ]:
FINAL_DIR.mkdir(parents=True, exist_ok=True)
final_scorer = CalibratedFraudPipeline(
    base_pipeline=selected_pipeline,
    calibrator=selected_calibrator,
    feature_names=tuple(core_features),
    model_version=MODEL_VERSION,
    calibration_method=selected_calibration_method,
)

threshold_policy = {
    "model_version": MODEL_VERSION,
    "calibration_method": selected_calibration_method,
    "selected_threshold_probability": selected_threshold,
    "selected_threshold_risk_score": 100 * selected_threshold,
    "medium_threshold_probability": medium_threshold,
    "critical_threshold_probability": critical_threshold,
    "analyst_capacity_per_1000": ANALYST_CAPACITY_PER_1000,
    "policy_selected_under_relaxed_gate": bool(policy_selected_under_relaxed_gate),
    "hard_rule_logic": "is_new_device AND is_new_beneficiary AND (is_high_balance_drain OR is_high_limit_usage)",
    "expected_cost_assumptions": {
        "cost_per_alert_vnd": COST_PER_ALERT_VND,
        "missed_fraud_loss_multiplier": MISSED_FRAUD_LOSS_MULTIPLIER,
    },
}

policy_lock_manifest = {
    **model_lock,
    "model_version": MODEL_VERSION,
    "calibration_method": selected_calibration_method,
    "threshold": selected_threshold,
    "critical_threshold": critical_threshold,
    "risk_bands": decision_bands.to_dict("records"),
    "alert_capacity_per_1000": ANALYST_CAPACITY_PER_1000,
    "locked_at_utc": datetime.now(timezone.utc).isoformat(),
    "test_opened": False,
}

joblib.dump(final_scorer, FINAL_DIR / "final_pipeline.joblib")
joblib.dump({"method": selected_calibration_method, "calibrator": selected_calibrator}, FINAL_DIR / "calibration_model.joblib")
shutil.copy2(SRC_DIR / "scoring_utils.py", FINAL_DIR / "scoring_utils.py")
(FINAL_DIR / "threshold_policy.json").write_text(json.dumps(threshold_policy, ensure_ascii=False, indent=2), encoding="utf-8")
(FINAL_DIR / "policy_lock_manifest.json").write_text(json.dumps(policy_lock_manifest, ensure_ascii=False, indent=2), encoding="utf-8")
decision_bands.to_csv(FINAL_DIR / "decision_bands.csv", index=False, encoding="utf-8-sig")
calibration_comparison.to_csv(FINAL_DIR / "calibration_comparison.csv", index=False)
threshold_candidates.to_csv(FINAL_DIR / "validation_threshold_candidates.csv", index=False)
strategy_comparison.to_csv(FINAL_DIR / "validation_strategy_comparison.csv", index=False)

assert joblib.load(FINAL_DIR / "final_pipeline.joblib").model_version == MODEL_VERSION
print("POLICY LOCKED — test_opened=false")
display(pd.DataFrame([policy_lock_manifest]).T.rename(columns={0: "locked_value"}))

<a id="8"></a>
## 8. Mở holdout test đúng quy trình

Đây là lần đầu notebook đọc nội dung `test.parquet`. Policy lock được đọc lại và kiểm tra `test_opened=false`; test chỉ được score bằng scorer đã khóa, không có `.fit()` nào từ đây trở xuống.

In [ ]:
pretest_lock = json.loads((FINAL_DIR / "policy_lock_manifest.json").read_text(encoding="utf-8"))
assert pretest_lock["test_opened"] is False
assert pretest_lock["pipeline_sha256"] == sha256_file(selected_path)
assert pretest_lock["feature_order_sha256"] == hashlib.sha256("\n".join(core_features).encode()).hexdigest()

df_test = pd.read_parquet(test_path)
assert set(df_test["simulation_run_id"].unique()) == set(split_manifest["test_set"]["runs"])
assert not (set(core_features) - set(df_test.columns))
assert not (required_audit - set(df_test.columns))
assert set(df_test["account_id"]).isdisjoint(df_validation["account_id"])
assert set(df_test["customer_id"]).isdisjoint(df_validation["customer_id"])

df_test["fraud_score_uncalibrated"] = selected_pipeline.predict_proba(df_test[core_features])[:, 1]
df_test["fraud_probability"] = final_scorer.predict_fraud_probability(df_test)
df_test["model_risk_score"] = 100.0 * df_test["fraud_probability"]
df_test["model_alert"] = df_test["fraud_probability"].ge(selected_threshold).astype(int)
df_test["risk_band"] = assign_band(df_test["fraud_probability"])
df_test["hard_rule_alert"] = (
    df_test["is_new_device"].astype(bool)
    & df_test["is_new_beneficiary"].astype(bool)
    & (df_test["is_high_balance_drain"].astype(bool) | df_test["is_high_limit_usage"].astype(bool))
).astype(int)
df_test["hybrid_alert"] = (df_test["model_alert"].eq(1) | df_test["hard_rule_alert"].eq(1)).astype(int)
df_test["decision"] = np.select(
    [df_test["hard_rule_alert"].eq(1) | df_test["risk_band"].eq("CRITICAL"), df_test["risk_band"].eq("HIGH"), df_test["risk_band"].eq("MEDIUM")],
    ["HOLD_AND_ALERT", "CHALLENGE_OR_ALERT", "REVIEW_CONTEXT"],
    default="APPROVE",
)
print(f"TEST OPENED — {len(df_test):,} rows | runs={df_test['simulation_run_id'].unique().tolist()}")

<a id="9"></a>
## 9. Metric cuối trên test

Metric test là con số nghiệm thu cuối: ranking, calibration, threshold, hard-negative, scenario và fraud amount. Không dùng các kết quả này để thay đổi policy hiện tại.

In [ ]:
y_test = df_test["target_fraud"].astype(int).to_numpy()
w_test = df_test["sample_weight"].to_numpy(dtype=float)
test_score = df_test["fraud_probability"].to_numpy()
test_pred = df_test["model_alert"].to_numpy()

test_policy_metric = policy_metrics(df_test, "fraud_probability", test_pred, "locked_model_policy", selected_threshold)
test_metric_record = {
    "model_version": MODEL_VERSION,
    "simulation_runs": ",".join(sorted(df_test["simulation_run_id"].unique())),
    "rows": len(df_test),
    "fraud_rows": int(y_test.sum()),
    "pr_auc_event_weighted": average_precision_score(y_test, test_score, sample_weight=w_test),
    "pr_auc_transaction": average_precision_score(y_test, test_score),
    "roc_auc": roc_auc_score(y_test, test_score, sample_weight=w_test),
    "brier_score_calibrated": brier_score_loss(y_test, test_score, sample_weight=w_test),
    "log_loss_calibrated": log_loss(y_test, np.column_stack([1-test_score, test_score]), sample_weight=w_test, labels=[0, 1]),
    "ece_10_bins": weighted_ece(y_test, test_score, w_test, n_bins=10),
    **{key: value for key, value in test_policy_metric.items() if key not in {"policy"}},
}
test_metrics = pd.DataFrame([test_metric_record])
test_scenario_recall = scenario_recall_table(df_test, "model_alert")

tn, fp, fn, tp = confusion_matrix(y_test, test_pred, labels=[0, 1]).ravel()
kpi_cards([
    {"label": "Test PR-AUC (event)", "value": f"{test_metric_record['pr_auc_event_weighted']:.4f}", "color": PALETTE["primary"]},
    {"label": "Recall @ locked threshold", "value": f"{test_metric_record['recall_event_weighted']:.1%}", "color": PALETTE["danger"]},
    {"label": "Alerts / 1.000", "value": f"{test_metric_record['alerts_per_1000']:.1f}", "color": PALETTE["accent"]},
    {"label": "Hard-negative alert", "value": f"{test_metric_record['hard_negative_alert_rate']:.1%}", "color": PALETTE["blue"]},
])
display(test_metrics.T.rename(columns={0: "test_value"}))
display(test_scenario_recall.style.format({"recall": "{:.2%}", "event_weight": "{:.2f}"}))
print(f"Confusion matrix: TN={tn:,}, FP={fp:,}, FN={fn:,}, TP={tp:,}")

### Scenario recall — model có bỏ quên một kiểu fraud cụ thể không?

Worst-scenario recall là guardrail quan trọng hơn chỉ nhìn metric tổng thể, đặc biệt với dữ liệu synthetic có các template phân tách mạnh.

In [ ]:
fig, ax = plt.subplots(figsize=(8.5, 4.8))
colors = [PALETTE["danger"] if value < MIN_WORST_SCENARIO_RECALL else PALETTE["primary"] for value in test_scenario_recall["recall"]]
ax.bar(test_scenario_recall["scenario_code"], test_scenario_recall["recall"], color=colors)
ax.axhline(MIN_WORST_SCENARIO_RECALL, linestyle="--", color=PALETTE["neutral"], label="Minimum guardrail")
clean_ax(ax, "Recall theo fraud scenario trên test", "Scenario", "Recall", "Policy đã khóa; không tối ưu lại theo biểu đồ này")
ax.set_ylim(0, 1.05)
ax.yaxis.set_major_formatter(lambda x, pos: f"{x:.0%}")
ax.legend()
plt.show()

<a id="10"></a>
## 10. Error analysis

Liệt kê false positive/false negative và driver tuyến tính lớn nhất. Đây là chẩn đoán cho model version kế tiếp; không sửa feature, hệ số hoặc threshold hiện tại từ test.

In [ ]:
preprocessor = selected_pipeline.named_steps["preprocessor"]
classifier = selected_pipeline.named_steps["model"]
processed_names = np.asarray(preprocessor.get_feature_names_out(), dtype=str)
transformed_test = np.asarray(preprocessor.transform(df_test[core_features]))
contributions = transformed_test * classifier.coef_[0]
top_positive_idx = contributions.argmax(axis=1)
top_negative_idx = contributions.argmin(axis=1)

df_test["top_positive_driver"] = processed_names[top_positive_idx]
df_test["top_positive_contribution"] = contributions[np.arange(len(df_test)), top_positive_idx]
df_test["top_negative_driver"] = processed_names[top_negative_idx]
df_test["top_negative_contribution"] = contributions[np.arange(len(df_test)), top_negative_idx]
df_test["error_type"] = np.select(
    [(df_test["target_fraud"].eq(0) & df_test["model_alert"].eq(1)), (df_test["target_fraud"].eq(1) & df_test["model_alert"].eq(0))],
    ["FALSE_POSITIVE", "FALSE_NEGATIVE"],
    default="CORRECT",
)
df_test["distance_to_threshold"] = df_test["fraud_probability"] - selected_threshold

error_columns = [
    "transaction_id", "account_id", "customer_id", "simulation_run_id", "txn_time_utc",
    "scenario_code", "hard_negative", "amount_num", "channel", "customer_segment",
    "account_type", "kyc_level", "fraud_probability", "model_risk_score", "risk_band",
    "model_alert", "error_type", "distance_to_threshold", "top_positive_driver",
    "top_positive_contribution", "top_negative_driver", "top_negative_contribution",
]
error_analysis = df_test.loc[df_test["error_type"].ne("CORRECT"), error_columns].copy()
error_summary = error_analysis.groupby(["error_type", "hard_negative"], dropna=False).agg(
    transactions=("transaction_id", "size"),
    total_amount=("amount_num", "sum"),
    median_score=("fraud_probability", "median"),
).reset_index()
display(error_summary)
display(error_analysis.sort_values("distance_to_threshold", key=abs).head(15))

<a id="11"></a>
## 11. Ổn định theo nhóm

Đánh giá model theo run, channel, segment, account type, KYC, scenario và population. Nhóm một lớp không có PR-AUC sẽ để `NaN` thay vì tạo metric giả.

In [ ]:
df_policy_stability = df_policy.copy()
df_policy_stability["evaluation_split"] = "validation_policy"
df_test_stability = df_test.copy()
df_test_stability["evaluation_split"] = "test"
stability_source = pd.concat([df_policy_stability, df_test_stability], ignore_index=True, sort=False)
stability_source["population"] = np.select(
    [stability_source["target_fraud"].eq(1), stability_source["hard_negative"].eq(1)],
    ["fraud", "hard_negative"],
    default="background_normal",
)


def grouped_stability(frame, dimension):
    rows = []
    for (split_name, group_value), group in frame.groupby(["evaluation_split", dimension], dropna=False):
        y = group["target_fraud"].astype(int).to_numpy()
        score = group["fraud_probability"].to_numpy()
        pred = group["model_alert"].to_numpy()
        weight = group["sample_weight"].to_numpy(dtype=float)
        rows.append({
            "evaluation_split": split_name,
            "dimension": dimension,
            "group": str(group_value),
            "rows": len(group),
            "fraud_rows": int(y.sum()),
            "pr_auc_event_weighted": safe_auc(average_precision_score, y, score, weight),
            "recall_event_weighted": recall_score(y, pred, sample_weight=weight, zero_division=0) if y.sum() else np.nan,
            "precision_event_weighted": precision_score(y, pred, sample_weight=weight, zero_division=0),
            "alert_rate": pred.mean(),
        })
    return rows


stability_rows = []
for dimension in ["simulation_run_id", "channel", "customer_segment", "account_type", "kyc_level", "scenario_code", "population"]:
    stability_rows.extend(grouped_stability(stability_source, dimension))
stability_metrics = pd.DataFrame(stability_rows)
display(stability_metrics.sort_values(["dimension", "evaluation_split", "group"]).head(30).style.format({
    "pr_auc_event_weighted": "{:.4f}", "recall_event_weighted": "{:.2%}",
    "precision_event_weighted": "{:.2%}", "alert_rate": "{:.2%}",
}))

<a id="12"></a>
## 12. Đóng gói bàn giao SAS

Xuất schema 34 input theo đúng thứ tự, mapping Python–SAS, golden cases đại diện các band và final manifest. Golden cases chứa input + expected score để so sai số Python/SAS; tolerance là contract kiểm thử, không phải threshold fraud.

In [ ]:
def dtype_contract(series):
    if pd.api.types.is_bool_dtype(series) or pd.api.types.is_integer_dtype(series):
        return "INTEGER"
    if pd.api.types.is_numeric_dtype(series):
        return "DOUBLE"
    return "VARCHAR"


sas_input_schema = pd.DataFrame([
    {
        "input_order": idx + 1,
        "feature_name": feature,
        "python_dtype": str(df_validation[feature].dtype),
        "sas_type": dtype_contract(df_validation[feature]),
        "required": True,
        "nullable_observed": bool(df_validation[feature].isna().any()),
        "role": "categorical" if feature in selected_pipeline.named_steps["preprocessor"].transformers_[1][2] else "numeric",
    }
    for idx, feature in enumerate(core_features)
])

sas_feature_mapping = feature_registry.set_index("feature_name").loc[core_features].reset_index()
sas_feature_mapping.insert(0, "input_order", range(1, len(sas_feature_mapping) + 1))
sas_feature_mapping = sas_feature_mapping.merge(
    sas_input_schema[["feature_name", "python_dtype", "sas_type", "required", "nullable_observed", "role"]],
    on="feature_name", how="left", validate="one_to_one",
)

golden_parts = []
for band in ["LOW", "MEDIUM", "HIGH", "CRITICAL"]:
    candidates = df_test.loc[df_test["risk_band"].eq(band)].sort_values("fraud_probability")
    if len(candidates):
        positions = sorted(set([0, len(candidates) // 2, len(candidates) - 1]))
        golden_parts.append(candidates.iloc[positions])
golden = pd.concat(golden_parts, ignore_index=True).drop_duplicates("transaction_id")
golden_columns = [
    "transaction_id", *core_features, "fraud_score_uncalibrated", "fraud_probability",
    "model_risk_score", "risk_band", "decision",
]
golden_scoring_cases = golden[golden_columns].copy()
golden_scoring_cases["probability_tolerance"] = 1e-6
golden_scoring_cases["model_version"] = MODEL_VERSION

test_prediction_columns = [
    "transaction_id", "account_id", "customer_id", "simulation_run_id", "txn_time_utc",
    "scenario_code", "target_fraud", "hard_negative", "amount_num",
    "fraud_score_uncalibrated", "fraud_probability", "model_risk_score",
    "risk_band", "model_alert", "hybrid_alert", "decision",
]

test_metrics.to_csv(FINAL_DIR / "test_metrics.csv", index=False)
df_test[test_prediction_columns].to_parquet(FINAL_DIR / "test_predictions.parquet", index=False)
test_scenario_recall.to_csv(FINAL_DIR / "test_scenario_recall.csv", index=False)
error_analysis.to_csv(FINAL_DIR / "error_analysis.csv", index=False, encoding="utf-8-sig")
stability_metrics.to_csv(FINAL_DIR / "stability_metrics.csv", index=False)
sas_input_schema.to_csv(FINAL_DIR / "sas_input_schema.csv", index=False, encoding="utf-8-sig")
sas_feature_mapping.to_csv(FINAL_DIR / "sas_feature_mapping.csv", index=False, encoding="utf-8-sig")
golden_scoring_cases.to_csv(FINAL_DIR / "golden_scoring_cases.csv", index=False, encoding="utf-8-sig")

final_artifact_names = [
    "final_pipeline.joblib", "calibration_model.joblib", "threshold_policy.json",
    "decision_bands.csv", "test_metrics.csv", "test_predictions.parquet",
    "test_scenario_recall.csv", "error_analysis.csv", "sas_input_schema.csv",
    "sas_feature_mapping.csv", "golden_scoring_cases.csv", "stability_metrics.csv",
    "scoring_utils.py",
]
final_model_manifest = {
    **policy_lock_manifest,
    "test_opened": True,
    "test_opened_at_utc": datetime.now(timezone.utc).isoformat(),
    "test_runs": sorted(df_test["simulation_run_id"].unique().tolist()),
    "test_metrics": {k: (v.item() if hasattr(v, "item") else v) for k, v in test_metric_record.items()},
    "data_limitations": training_manifest.get("upstream_limitations", []) + [
        "All reported performance is measured on synthetic simulation runs.",
        "Expected cost uses configurable assumptions and is not an approved finance estimate.",
        "Test error analysis must not feed back into this model version.",
    ],
    "artifacts": final_artifact_names,
}
(FINAL_DIR / "final_model_manifest.json").write_text(json.dumps(final_model_manifest, ensure_ascii=False, indent=2), encoding="utf-8")

expected_artifacts = [FINAL_DIR / name for name in final_artifact_names] + [FINAL_DIR / "final_model_manifest.json"]
assert all(path.exists() and path.stat().st_size > 0 for path in expected_artifacts)
assert len(sas_input_schema) == EXPECTED_RAW_FEATURES
assert sas_input_schema["feature_name"].tolist() == core_features
reloaded_scorer = joblib.load(FINAL_DIR / "final_pipeline.joblib")
reloaded_score = reloaded_scorer.predict_fraud_probability(golden_scoring_cases[core_features])
assert np.allclose(reloaded_score, golden_scoring_cases["fraud_probability"], atol=1e-12)

artifact_inventory = pd.DataFrame({
    "artifact": [path.name for path in expected_artifacts],
    "bytes": [path.stat().st_size for path in expected_artifacts],
    "sha256": [sha256_file(path) for path in expected_artifacts],
})
display(artifact_inventory)
print("SAS HANDOFF READY" if not policy_selected_under_relaxed_gate else "SAS HANDOFF PROVISIONAL — policy gate đã được nới")

## Kết luận và handoff

- Model/preprocessing/34 CORE features được giữ nguyên từ Notebook 04.
- Calibration, threshold, bands và hybrid rule được chọn hoàn toàn trên validation trước khi test được đọc.
- Run 005 chỉ dùng một chiều để nghiệm thu và phân tích lỗi; không có feedback ngược vào model hiện tại.
- `final_model_manifest.json`, `sas_input_schema.csv`, `sas_feature_mapping.csv` và `golden_scoring_cases.csv` là contract chính cho bước đối soát Python–SAS.
- Alert Triage nằm ngoài notebook này và bắt đầu sau khi SAS scoring/decisioning tái tạo đúng golden cases.